In [ ]:
import os
import glob
import pandas as pd
from sentence_transformers import SentenceTransformer
from google.colab import drive

# 1. Mount Google Drive permanently
drive.mount('/content/drive')

# 2. Configuration & Paths
# Set the master directory for reading and saving data
DRIVE_PATH = "/content/drive/MyDrive/Colab Notebooks/data"
CHECKPOINT_DIR = f"{DRIVE_PATH}/cds_checkpoints"

# Ensure the checkpoint directory exists inside your data folder
os.makedirs(CHECKPOINT_DIR, exist_ok=True)



Mounted at /content/drive


In [ ]:
# Model and processing settings optimized for Colab Pro (A100/High-RAM)
MODEL_NAME = 'BAAI/bge-large-en-v1.5'
CHUNK_SIZE = 100000
BATCH_SIZE = 1024

# 3. Initialize Model
print(f"Loading {MODEL_NAME} onto GPU...")
model = SentenceTransformer(MODEL_NAME)

# 4. Load your actual data from the new DRIVE_PATH
print(f"Loading dataset from {DRIVE_PATH}...")
# Update "your_raw_data.csv" to the actual name of your file
data_file = f"{DRIVE_PATH}/combined_human_ai.csv"
df = pd.read_csv(data_file)

# Ensure all data in your target column is string format to prevent tokenization errors
# Update 'post_content' if your text column is named something else
texts_to_embed = df['post'].astype(str).tolist()

# 5. Processing Loop with Google Drive Checkpointing
print(f"Starting processing of {len(texts_to_embed)} rows...")
chunk_index = 0

for start_idx in range(0, len(texts_to_embed), CHUNK_SIZE):
    end_idx = min(start_idx + CHUNK_SIZE, len(texts_to_embed))
    current_chunk_texts = texts_to_embed[start_idx:end_idx]

    print(f"\nProcessing rows {start_idx} to {end_idx}...")

    # Generate embeddings on the GPU
    chunk_embeddings = model.encode(
        current_chunk_texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    # Slice the dataframe and attach embeddings
    df_chunk = df.iloc[start_idx:end_idx].copy()
    df_chunk['embedding'] = chunk_embeddings.tolist()

    # Save checkpoint directly to the nested checkpoints folder
    filename = f"{CHECKPOINT_DIR}/embeddings_part_{chunk_index}.pkl"
    df_chunk.to_pickle(filename)
    print(f"Saved checkpoint securely to: {filename}")

    chunk_index += 1

# 6. Combine all checkpoints into final file on Drive
print("\nMerging all checkpoints from Google Drive...")
all_files = sorted(glob.glob(f"{CHECKPOINT_DIR}/*.pkl"))

# Load and concatenate all chunks
df_final = pd.concat([pd.read_pickle(f) for f in all_files], ignore_index=True)

# Save the final master file to the root of your target data folder
final_path = f"{DRIVE_PATH}/final_600k_bge_large_embeddings.pkl"
df_final.to_pickle(final_path)

print(f"\nSuccess! Final dataset saved to: {final_path}")
print(f"Final dataset shape: {df_final.shape}")
print(f"Vector dimensions: {len(df_final['embedding'].iloc[0])}")

Loading BAAI/bge-large-en-v1.5 onto GPU...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

Saved checkpoint securely to: /content/drive/MyDrive/Colab Notebooks/data/cds_checkpoints/embeddings_part_3.pkl

Processing rows 400000 to 500000...


Batches:   0%|          | 0/98 [00:00<?, ?it/s]

Saved checkpoint securely to: /content/drive/MyDrive/Colab Notebooks/data/cds_checkpoints/embeddings_part_4.pkl

Processing rows 500000 to 563088...


Batches:   0%|          | 0/62 [00:00<?, ?it/s]

Saved checkpoint securely to: /content/drive/MyDrive/Colab Notebooks/data/cds_checkpoints/embeddings_part_5.pkl

Merging all checkpoints from Google Drive...

Success! Final dataset saved to: /content/drive/MyDrive/Colab Notebooks/data/final_600k_bge_large_embeddings.pkl
Final dataset shape: (563088, 11)
Vector dimensions: 1024
